# Heart Sound Analysis -- Automated Benchmark & Audit Pipeline
### Leakage-Aware Evaluation, 5 Architectural Ablations, Explanation Audits & Paper Deliverables

**Instructions:**
1. Go to **Runtime -> Change runtime type** and ensure **T4 GPU** is selected.
2. Click **Runtime -> Run all** (or press `Ctrl + F9`).
3. The notebook will automatically execute the entire 24-experiment benchmark, audit explanation faithfulness, compile all LaTeX tables and macros, and download the results.


## 0. Environment Setup & GPU Verification

In [ ]:
# Check GPU availability (CUDA or Apple Silicon MPS)
import sys
import shutil
import torch

print(f'PyTorch Version: {torch.__version__}')
cuda_avail = torch.cuda.is_available()
mps_avail = hasattr(torch.backends, 'mps') and torch.backends.mps.is_available()
print(f'CUDA Available: {cuda_avail}')
print(f'Apple Silicon MPS Available: {mps_avail}')

if cuda_avail:
    print(f'GPU Device: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB')
    if shutil.which('nvidia-smi'):
        !nvidia-smi
elif mps_avail:
    print('Using Apple Silicon Metal Performance Shaders (MPS) GPU acceleration.')
else:
    print('Running on CPU.')


In [ ]:
# Install required Python dependencies
%pip install -q librosa scipy numpy pandas torch torchvision tqdm shap h5py pytest

## 1. Repository Setup & Drive Mount (Optional)

In [ ]:
import os
from pathlib import Path

# 1. Mount Google Drive if running in Colab (browser or VS Code Colab extension)
if os.path.exists('/content'):
    if not Path('/content/drive/MyDrive').exists():
        from google.colab import drive
        drive.mount('/content/drive')
    print('Google Drive mounted.')
    project_dir = Path('/content/drive/MyDrive/heart-sound-analysis')
else:
    print('Running locally on MacBook.')
    project_dir = Path.cwd()

# 2. Set active working directory directly to the project folder
os.chdir(str(project_dir))
print(f'Active working directory: {Path.cwd()}')

## 2. Dataset Verification & Extraction

In [ ]:
from pathlib import Path

data_dir = Path("physionet_2016")
spec_dir = data_dir / "spectrograms"
meta_csv = data_dir / "metadata.csv"

num_specs = len(list(spec_dir.glob("*.npy"))) if spec_dir.exists() else 0
print(f"Spectrograms available : {num_specs} / 3541")
print(f"Metadata CSV exists    : {meta_csv.exists()}")
print(f"Dataset status         : {'READY' if num_specs >= 3541 and meta_csv.exists() else 'CHECK PATH'}")

## 3. Generate Splits & Run Duplicate Leakage Audit

In [ ]:
# Generate deduplicated (3 seeds) and LODO splits
import sys
py_exec = sys.executable

!{py_exec} 'Review 3/make_splits.py' --mode dedup --seeds 42 43 44
!{py_exec} 'Review 3/make_splits.py' --mode lodo --holdouts a b e f --seed 42

# Check twin duplicate performance vs unique test accuracy
!{py_exec} 'Review 3/verify_duplicates.py'


## 4. Run All 24 Benchmark Experiments Sequentially

In [ ]:
# Dry-run verification (24 planned runs)
import sys
py_exec = sys.executable
!{py_exec} 'Review 3/run_all.py' --dry-run

# Execute full 24-run benchmark (resume-safe: skips already completed runs)
!{py_exec} 'Review 3/run_all.py'


## 5. Run Population-Level Explanation Audit Suite

In [ ]:
# Execute audit on primary model: dedup_se_mha_seed42
import sys
py_exec = sys.executable
!{py_exec} 'Review 3/audit/run_audit.py' --tag dedup_se_mha_seed42


## 6. Aggregate Deliverables & Compile LaTeX Paper

In [ ]:
# Aggregate all JSON artifacts into LaTeX tables, macros, and findings
import sys, shutil
py_exec = sys.executable
!{py_exec} 'Review 3/aggregate_results.py'

# Run unit tests to verify consistency
!{py_exec} -m pytest -q

# Compile PDF using pdflatex if available
if shutil.which('pdflatex'):
    %cd 'Paper Writing'
    !pdflatex -interaction=nonstopmode paper.tex
    !bibtex paper
    !pdflatex -interaction=nonstopmode paper.tex
    !pdflatex -interaction=nonstopmode paper.tex
    %cd ..
elif shutil.which('apt-get'):
    !apt-get update -qq && apt-get install -y -qq texlive-latex-base texlive-fonts-recommended texlive-fonts-extra texlive-latex-extra
    %cd 'Paper Writing'
    !pdflatex -interaction=nonstopmode paper.tex
    !bibtex paper
    !pdflatex -interaction=nonstopmode paper.tex
    !pdflatex -interaction=nonstopmode paper.tex
    %cd ..
else:
    print('pdflatex not found on system. Skipping PDF compilation.')


## 7. Download Results Archive

In [ ]:
from pathlib import Path

# Create zip of all experimental results and the compiled paper PDF
output_archive = 'benchmark_results_and_paper.zip'
!zip -q -r benchmark_results_and_paper.zip results 'Paper Writing/generated' 
print(f'Created archive: {output_archive}')

try:
    from google.colab import files
    files.download(output_archive)
except Exception:
    print(f'Saved locally at: {Path.cwd() / output_archive}')
